In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from collections import defaultdict

rng = np.random.default_rng(seed=42)

In [2]:
def make_dataset(n, d, rng):
    """Generate n items uniformly in [-1, 1]^d."""
    return rng.uniform(-1, 1, size=(n, d))

# quick check
X = make_dataset(n=100, d=5, rng=rng)
print(f"X shape: {X.shape}")
print(f"X range: [{X.min():.3f}, {X.max():.3f}]")

X shape: (100, 5)
X range: [-0.989, 0.998]


In [3]:
def gamma_ckl_prob(x_i, x_j, x_t, gamma):
    """P(oracle picks i | query is (i,j), target is t) under γ-CKL.
    
    p = ||x_j - x_t||^γ / (||x_i - x_t||^γ + ||x_j - x_t||^γ)
    """
    d_i = np.linalg.norm(x_i - x_t)
    d_j = np.linalg.norm(x_j - x_t)
    # if either distance is 0, that item is the target — return decisively
    if d_i == 0: return 1.0
    if d_j == 0: return 0.0
    return (d_j ** gamma) / (d_i ** gamma + d_j ** gamma)

def query_oracle(x_i, x_j, x_t, gamma, rng):
    """Sample an oracle answer. Returns 0 (chose i) or 1 (chose j)."""
    p_i = gamma_ckl_prob(x_i, x_j, x_t, gamma)
    return 0 if rng.random() < p_i else 1

# smoke test: oracle should almost always pick the closer item when γ is high
X_test = make_dataset(n=3, d=2, rng=rng)
x_i, x_j, x_t = X_test[0], X_test[1], X_test[2]
p = gamma_ckl_prob(x_i, x_j, x_t, gamma=5)
print(f"distances: i={np.linalg.norm(x_i-x_t):.3f}, j={np.linalg.norm(x_j-x_t):.3f}")
print(f"P(pick i) = {p:.3f}")
# simulate 1000 queries, see the empirical rate
answers = [query_oracle(x_i, x_j, x_t, gamma=5, rng=rng) for _ in range(1000)]
print(f"empirical P(pick i) = {answers.count(0)/1000:.3f}")

distances: i=0.466, j=0.997
P(pick i) = 0.978
empirical P(pick i) = 0.969


In [4]:
def sample_mirror(X, P, used, r=2.0):
    """Pick next query (i, j) using the SAMPLEMIRROR heuristic.
    
    1. Compute weighted mean μ and weighted covariance Σ over items.
    2. Find eigenvector v_max of Σ with largest eigenvalue λ_max.
    3. Construct proto-query points on either side of μ along v_max.
    4. Find nearest unused items to each proto-query, weighted by posterior.
    
    Args:
        X: (n, d) item embeddings
        P: (n,) current posterior over items
        used: set of item indices already queried
        r: scale factor for proto-query distance from μ
    
    Returns:
        (i, j): indices of the two items to query
    """
    n, d = X.shape
    
    # weighted mean and covariance
    mu = (P[:, None] * X).sum(axis=0)          # (d,)
    diff = X - mu                                # (n, d)
    Sigma = (P[:, None, None] * diff[:, :, None] * diff[:, None, :]).sum(axis=0)  # (d, d)
    
    # top eigenvector
    eigvals, eigvecs = np.linalg.eigh(Sigma)   # ascending order
    lam_max = eigvals[-1]
    v_max = eigvecs[:, -1]
    
    # proto-query points
    z1 = mu + r * np.sqrt(max(lam_max, 1e-12)) * v_max
    z2 = mu - r * np.sqrt(max(lam_max, 1e-12)) * v_max
    
    # find nearest items weighted by P, excluding used
    def nearest(z):
        # score = P_k * ||x_k - z||^2 — argmin, ignoring used
        d2 = np.sum((X - z) ** 2, axis=1)
        score = P * d2
        # mask used
        for u in used:
            score[u] = np.inf
        return int(np.argmin(score))
    
    i = nearest(z1)
    # ensure j != i by masking i temporarily
    used_plus_i = used | {i}
    def nearest_excl(z, excl):
        d2 = np.sum((X - z) ** 2, axis=1)
        score = P * d2
        for u in excl:
            score[u] = np.inf
        return int(np.argmin(score))
    j = nearest_excl(z2, used_plus_i)
    
    return i, j

In [5]:
def bayes_update(X, P, i, j, y, gamma, eps=1e-12):
    """Update posterior P after observing oracle answer y ∈ {i, j}.
    
    P_k ← P_k · P(y | i, j, x_k) / Z
    """
    n = len(P)
    # compute likelihood for each candidate target x_k
    d_i = np.linalg.norm(X - X[i], axis=1)    # (n,) distance from each item to i
    d_j = np.linalg.norm(X - X[j], axis=1)
    
    d_i_g = d_i ** gamma
    d_j_g = d_j ** gamma
    denom = d_i_g + d_j_g + eps
    
    if y == i:
        # oracle picked i: likelihood is P(pick i | x_k) = d_j^γ / (d_i^γ + d_j^γ)
        likelihood = d_j_g / denom
    else:
        likelihood = d_i_g / denom
    
    P_new = P * likelihood
    Z = P_new.sum()
    if Z < eps:
        # numerical fallback: keep the prior (shouldn't happen in practice)
        return P.copy()
    return P_new / Z

In [8]:
def gamma_ckl_search(X, target_idx, gamma, r=2.0, max_queries=200, rng=None):
    """Run γ-CKLSearch until target is in the query pair.
    
    Returns dict with query count, posterior history, and query sequence.
    """
    if rng is None:
        rng = np.random.default_rng()
    n = len(X)
    P = np.ones(n) / n
    used = set()
    x_t = X[target_idx]
    
    history = {"P_target": [], "queries": [], "argmax": []}
    
    for step in range(max_queries):
        i, j = sample_mirror(X, P, used, r=r)
        used |= {i, j}
        
        # oracle answer
        y_idx = query_oracle(X[i], X[j], x_t, gamma, rng)
        y = i if y_idx == 0 else j
        
        # log
        history["P_target"].append(P[target_idx])
        history["queries"].append((i, j))
        history["argmax"].append(int(np.argmax(P)))
        
        # stop if target is in the query
        if target_idx in (i, j):
            return {"found": True, "n_queries": step + 1, "history": history}
        
        # update belief
        P = bayes_update(X, P, i, j, y, gamma)
    
    return {"found": False, "n_queries": max_queries, "history": history}


# smoke test: single search
rng = np.random.default_rng(seed=0)
X = make_dataset(n=100, d=5, rng=rng)
target_idx = 42

result = gamma_ckl_search(X, target_idx, gamma=5, r=2.0, max_queries=50, rng=rng)
print(f"found: {result['found']}, queries: {result['n_queries']}")
print(f"P(target) trajectory (first 20 steps):")
for step, p in enumerate(result['history']['P_target'][:20]):
    argmax = result['history']['argmax'][step]
    marker = "  <-- target is argmax" if argmax == target_idx else ""
    print(f"  step {step:3d}: P[target] = {p:.4f}, argmax = {argmax}{marker}")

found: True, queries: 3
P(target) trajectory (first 20 steps):
  step   0: P[target] = 0.0100, argmax = 0
  step   1: P[target] = 0.0322, argmax = 23
  step   2: P[target] = 0.0620, argmax = 23


In [7]:

def sample_mirror(X, P, used, r=2.0, eps=1e-12):
    """Pick next query (i, j) using the SAMPLEMIRROR heuristic.
    
    Favors items that are BOTH close to the proto-query point AND high-probability.
    """
    n, d = X.shape
    
    mu = (P[:, None] * X).sum(axis=0)
    diff = X - mu
    Sigma = (P[:, None, None] * diff[:, :, None] * diff[:, None, :]).sum(axis=0)
    
    eigvals, eigvecs = np.linalg.eigh(Sigma)
    lam_max = eigvals[-1]
    v_max = eigvecs[:, -1]
    
    z1 = mu + r * np.sqrt(max(lam_max, 1e-12)) * v_max
    z2 = mu - r * np.sqrt(max(lam_max, 1e-12)) * v_max
    
    def nearest(z, excl):
        d2 = np.sum((X - z) ** 2, axis=1)
        # score: high when close AND probable. Lower = better.
        score = d2 / (P + eps)
        for u in excl:
            score[u] = np.inf
        return int(np.argmin(score))
    
    i = nearest(z1, used)
    j = nearest(z2, used | {i})
    
    return i, j

In [9]:
rng = np.random.default_rng(seed=0)
X = make_dataset(n=100, d=5, rng=rng)

results = []
for target_idx in range(20):  # test 20 different targets
    result = gamma_ckl_search(X, target_idx, gamma=5, r=2.0, max_queries=100, rng=rng)
    results.append(result['n_queries'])
    print(f"target {target_idx:3d}: {result['n_queries']} queries, found={result['found']}")

print(f"\nmean: {np.mean(results):.2f}, median: {np.median(results):.1f}, max: {max(results)}")

target   0: 11 queries, found=True
target   1: 7 queries, found=True
target   2: 10 queries, found=True
target   3: 13 queries, found=True
target   4: 5 queries, found=True
target   5: 6 queries, found=True
target   6: 6 queries, found=True
target   7: 5 queries, found=True
target   8: 5 queries, found=True
target   9: 6 queries, found=True
target  10: 7 queries, found=True
target  11: 6 queries, found=True
target  12: 9 queries, found=True
target  13: 13 queries, found=True
target  14: 5 queries, found=True
target  15: 2 queries, found=True
target  16: 5 queries, found=True
target  17: 8 queries, found=True
target  18: 3 queries, found=True
target  19: 5 queries, found=True

mean: 6.85, median: 6.0, max: 13
